# 02 — Steering Vectors

**Objetivo (días 5-6):** calcular e inyectar un steering vector en GPT-2 y comparar outputs con y sin steering.

**Entregable:** notebook funcional con steering vector de escepticismo inyectado. Output antes/después visible con mínimo 5 prompts.

**Prerequisito:** haber completado el notebook 01 — se asume que sabes qué es el residual stream, `run_with_cache()` y los hooks.

## Diccionario de conceptos

---

**Representación lineal de features**

Hipótesis central del activation steering: los modelos de lenguaje codifican conceptos como *direcciones* en el espacio de activaciones. Si el modelo tiene un concepto de escepticismo, existe una dirección en R^768 tal que moverse en esa dirección hace que el modelo procese el texto de forma más escéptica. Esta hipótesis está respaldada empíricamente (Turner et al. 2023, Elhage et al. 2022) y justifica que una simple suma vectorial pueda modificar el comportamiento del modelo.

---

**Espacio de activaciones**

El espacio vectorial de dimensión `d_model` (768 en GPT-2) donde viven las representaciones internas del modelo. Cada token en cada capa tiene una representación — un punto en este espacio. Los conceptos son regiones o direcciones dentro de él. La interpretabilidad mecanicista estudia la geometría de este espacio.

---

**Superposición (superposition)**

Fenómeno por el que los modelos codifican más features que dimensiones. Un espacio de 768 dimensiones puede representar miles de conceptos usando direcciones casi-ortogonales. Esto implica que los conceptos no ocupan dimensiones individuales sino combinaciones de ellas. La representación lineal funciona gracias a la superposición: cada dirección puede ser suficientemente ortogonal a las demás para ser distinguible. (Elhage et al. 2022)

---

**Contrastive Activation Engineering (ActAdd)**

El método de Turner et al. 2023 para calcular steering vectors. La intuición: si quiero encontrar la dirección del concepto escepticismo, genero activaciones con prompts que expresan certeza y prompts que expresan duda. La diferencia entre los centroides de ambos grupos es una estimación de la dirección del concepto en el espacio de activaciones.

---

**Prompts contrastivos**

Pares de prompts que representan conceptos opuestos. Un conjunto expresa el concepto positivo (lo que queremos añadir) y otro el negativo (de lo que queremos alejarnos). Usar múltiples prompts por concepto y promediar reduce el ruido específico de cada frase y captura mejor la dirección del concepto.

---

**Centroide**

La media aritmética de un conjunto de vectores. En este contexto: el punto promedio en el espacio de activaciones de todos los prompts de un mismo concepto. El steering vector es la diferencia entre el centroide del concepto negativo y el centroide del concepto positivo.

---

**Steering vector**

Un vector en R^768 que apunta de un concepto a otro en el espacio de activaciones. Se calcula como `centroide_negativo - centroide_positivo`. Sumarlo al residual stream durante la inferencia empuja la representación activa hacia la zona del espacio asociada al concepto negativo.

---

**Alpha — coeficiente de intensidad**

Escalar que controla la magnitud del steering. El vector se suma como `alpha * steering_vector`. Un alpha muy pequeño produce un efecto imperceptible; un alpha muy grande rompe la coherencia del texto porque desplaza la representación demasiado lejos de las zonas que el modelo sabe interpretar. El rango óptimo depende de la capa y la norma del vector — típicamente entre 10 y 20 para GPT-2.

---

**Norma del steering vector**

La magnitud del vector (`torch.norm()`). Si es muy pequeña, los dos conceptos son casi indistinguibles para el modelo en esa capa — señal de que hay que probar otra capa. Si es muy grande, puede indicar que los prompts son semánticamente muy distintos por razones no relacionadas con el concepto buscado.

---

**Inyección en el residual stream**

El mecanismo de steering: sumar el vector al tensor del residual stream en una capa específica durante el forward pass. Las capas posteriores reciben ese tensor modificado y continúan el cómputo normalmente. El modelo no sabe que hubo intervención — procesa la información modificada como si fuera la representación natural del texto.

---

**Capa de inyección**

La capa del transformer donde se inyecta el steering vector. Las capas intermedias (8-12 de 12 en GPT-2) suelen dar los mejores resultados: las capas iniciales codifican sintaxis básica y aún no tienen semántica rica; las capas finales están enfocadas en la predicción del próximo token y el steering tiene menos efecto. El rango 8-12 es el punto donde el modelo tiene representaciones semánticas maduras.

---

**Generación autoregresiva**

El proceso de generar texto token a token: el modelo predice el siguiente token, lo añade a la secuencia, y repite. En cada paso se hace un forward pass completo. El hook de steering se activa en cada uno de esos forward passes, así que el efecto se aplica continuamente durante toda la generación.

---

**Hedging lingüístico**

Expresiones que indican incertidumbre o cautela: might, perhaps, it is unclear, some argue, allegedly, it's possible that. Es el efecto observable esperado de un steering vector de escepticismo que funciona correctamente.

---

**Reversibilidad**

Una ventaja clave del steering frente al fine-tuning: el efecto existe solo mientras el hook está registrado. Quitar el hook restaura el comportamiento original al instante, sin recargar el modelo. Los pesos nunca se modifican.

## 1. Setup

Este notebook construye sobre el 01. Mientras que en el 01 aprendimos a *leer* activaciones con `run_with_cache`, aquí aprendemos a *escribir* en ellas durante la inferencia.

El flujo completo:
1. Definir prompts contrastivos (certeza vs. escepticismo)
2. Extraer activaciones medias de cada conjunto en una capa intermedia
3. Calcular la dirección en el espacio de activaciones que separa ambos conceptos
4. Inyectar esa dirección durante la generación y observar el cambio de comportamiento

In [ ]:
!pip install transformer_lens -q

In [ ]:
import torch
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained('gpt2')
model.eval()

## 2. Prompts contrastivos

**Contrastive activation engineering** (Turner et al. 2023): la hipótesis central es que los conceptos en un LLM están codificados como *direcciones* en el espacio de activaciones. Para encontrar la dirección del escepticismo:

1. Genera activaciones con prompts que expresan **certeza** -> calcula su centroide en el espacio
2. Genera activaciones con prompts que expresan **duda** -> otro centroide
3. La diferencia entre ambos centroides es el vector que apunta de certeza a duda

El método asume que esta dirección es relativamente consistente entre distintos prompts del mismo concepto. Por eso usamos múltiples ejemplos y promediamos — un solo par de prompts podría capturar ruido específico de esas frases en lugar de la dirección del concepto.

Elegimos 3 prompts por concepto: suficiente para reducir el ruido sin necesitar un dataset grande.

In [ ]:
positive_prompts = [
    'I am certain that',
    'It is absolutely true that',
    'Without any doubt,',
]

negative_prompts = [
    'I doubt that',
    'I am not sure whether',
    'It is unclear whether',
]

## 3. Extraer activaciones medias por concepto

Dos decisiones de diseño que merece la pena entender:

**Que token tomamos:** el último token (`[0, -1, :]`). En modelos autorregresivos como GPT-2 el último token es el que va a predecir el siguiente, por lo tanto su representación en el residual stream integra toda la información contextual del prompt. Es el punto donde confluye el significado.

**Que capa usamos:** las capas intermedias (8-12 de 12 en GPT-2) suelen tener las representaciones semánticas más ricas. Las capas iniciales codifican sintaxis y tokens individuales. Las capas finales están enfocadas en la tarea de predicción del próximo token. El rango 8-12 es el punto dulce donde el modelo ha procesado suficiente contexto pero aún no ha colapsado hacia la distribución de salida.

In [ ]:
LAYER = 10  # probar capas 8-14

def get_mean_activation(prompts, layer):
    activations = []
    for prompt in prompts:
        tokens = model.to_tokens(prompt)
        _, cache = model.run_with_cache(tokens)
        # Tomamos el ultimo token del residual stream
        act = cache['resid_post', layer][0, -1, :]  # (d_model,)
        activations.append(act)
    return torch.stack(activations).mean(dim=0)

pos_mean = get_mean_activation(positive_prompts, LAYER)
neg_mean = get_mean_activation(negative_prompts, LAYER)

print('Shape activacion media:', pos_mean.shape)

## 4. Calcular el steering vector

**Geometria del steering vector:**

```
pos_mean (certeza) --------> neg_mean (escepticismo)
                       ^
                       |
              steering_vector = neg_mean - pos_mean
```

El vector apunta de certeza hacia escepticismo en el espacio de 768 dimensiones. Al sumarlo al residual stream durante la inferencia, trasladamos la representación activa hacia la zona del espacio donde el modelo vive cuando procesa escepticismo.

La **norma** del vector es un indicador de calidad: si es muy pequeña, los dos conceptos son casi indistinguibles para el modelo en esa capa. Si es muy grande, puede indicar que los prompts son semánticamente muy distintos por razones no relacionadas con el concepto.

In [ ]:
# Steering vector: direccion de positivo a negativo (escepticismo)
steering_vector = neg_mean - pos_mean
print('Steering vector shape:', steering_vector.shape)
print('Norma:', steering_vector.norm().item())

## 5. Inyectar el vector con un hook

**Mecanismo de inyeccion con hooks:**

El hook se registra en `blocks.{LAYER}.hook_resid_post`. En cada paso de generacion, justo despues de que la capa `LAYER` calcula su contribucion, el hook suma `ALPHA * steering_vector` al ultimo token del residual stream. Las capas posteriores reciben ese tensor modificado y continuan el computo normalmente — sin saber que hubo intervencion.

**Alpha — como elegirlo:**

| alpha | Efecto esperado |
|-------|------------------|
| 1-5   | Cambio sutil, dificil de detectar |
| 10-20 | Efecto claro manteniendo coherencia — zona objetivo |
| > 30  | Output incoherente o degenerado |

La escala optima depende de la norma del steering vector y de la capa. Una norma alta requiere un alpha mas bajo.

In [ ]:
ALPHA = 15  # probar 5, 10, 15, 20

def steering_hook(value, hook):
    value[:, -1, :] += ALPHA * steering_vector
    return value

def generate(prompt, max_new_tokens=30, use_steering=False):
    tokens = model.to_tokens(prompt)
    if use_steering:
        hooks = [(f'blocks.{LAYER}.hook_resid_post', steering_hook)]
        output = model.generate(tokens, max_new_tokens=max_new_tokens, hooks=hooks)
    else:
        output = model.generate(tokens, max_new_tokens=max_new_tokens)
    return model.to_string(output[0])

## 6. Comparativa — baseline vs steered

**Que buscar en los outputs:**

Un steering efectivo de escepticismo deberia producir hedging linguistico:
- Palabras como might, perhaps, it is unclear, some argue, allegedly
- Afirmaciones menos directas o mas condicionales
- Menciones explicitas de incertidumbre o posibles alternativas

**Si el output steered es incoherente o repetitivo:** alpha demasiado alto o capa demasiado superficial. Reduce alpha o prueba capas mas profundas (10-12).

**Si no ves ningun cambio:** alpha demasiado bajo o el steering vector no captura bien el concepto en esa capa. Aumenta alpha o experimenta con otras capas.

In [ ]:
test_prompts = [
    'Scientists have discovered that',
    'The new policy will',
    'According to experts,',
    'The results of the study show',
    'People believe that',
]

for prompt in test_prompts:
    baseline = generate(prompt, use_steering=False)
    steered  = generate(prompt, use_steering=True)
    print(f'PROMPT: {prompt}')
    print(f'  BASELINE : {baseline}')
    print(f'  STEERED  : {steered}')
    print()

## 7. Notas — que capa y que alpha funcionan mejor

Documenta tus resultados experimentales. Para cada combinacion capa/alpha, anota si el output es coherente y si el efecto es perceptible.

**Parametros recomendados para empezar:** capa 10, alpha 15. Desde ahi:
- Si no hay efecto: sube alpha a 20 o prueba capa 11
- Si el texto degenera: baja alpha a 10 o prueba capa 9

| Capa | Alpha | Coherencia (1-5) | Efecto visible | Observacion |
|------|-------|------------------|----------------|-------------|
| 10   | 15    |                  |                |             |
|      |       |                  |                |             |